In [ ]:
import glob
from scipy.io import loadmat

import os
from sklearn.metrics import mean_squared_error, mean_absolute_error,root_mean_squared_error
from scipy.cluster.hierarchy import fclusterdata

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import math
import sys

os.chdir('/Users/davidlee/Desktop/Sionna_Princeton')
sys.path.append(os.path.abspath('./src'))

In [28]:
# LOS or NLOS for measurement type
measurement_type1 = "NLOS"
measurement_type2 = "LOS"

In [29]:
def initialize_header(measurement_type):
    # Initialize header data structure
    header_dict = {"file_name":[], "mea_idx":[]}
    header_folder_path = "./data/measurements/Boulder_Downtown/Measurements/BoulderDowntown_28GHz_{}/Header files/".format(measurement_type)  
    header_mat_files = glob.glob(f"{header_folder_path}/*.mat")  
    print(header_mat_files[:3]) 
    return header_dict, header_folder_path, header_mat_files

In [30]:
NLOS_header_dict, NLOS_header_folder_path, NLOS_header_mat_files = initialize_header(measurement_type1)
LOS_header_dict, LOS_header_folder_path, LOS_header_mat_files = initialize_header(measurement_type2)

['./data/measurements/Boulder_Downtown/Measurements/BoulderDowntown_28GHz_NLOS/Header files/F0002150hdr.mat', './data/measurements/Boulder_Downtown/Measurements/BoulderDowntown_28GHz_NLOS/Header files/F0002142hdr.mat', './data/measurements/Boulder_Downtown/Measurements/BoulderDowntown_28GHz_NLOS/Header files/F0002174hdr.mat']
['./data/measurements/Boulder_Downtown/Measurements/BoulderDowntown_28GHz_LOS/Header files/F0001599hdr.mat', './data/measurements/Boulder_Downtown/Measurements/BoulderDowntown_28GHz_LOS/Header files/F0001587hdr.mat', './data/measurements/Boulder_Downtown/Measurements/BoulderDowntown_28GHz_LOS/Header files/F0001595hdr.mat']


In [31]:
def extract_header_info(header_dict, header_mat_files):
    # MATLAB to Python data conversion
    for mat_file_path in header_mat_files: 

        #Load the .mat file
        data = loadmat(mat_file_path)

        # Store basic file info
        header_dict["file_name"].append(os.path.basename(mat_file_path))
        header_dict["mea_idx"].append(os.path.basename(mat_file_path)[4:4+4])

        # Recursive struct parsing
        for key in data:
            if not key.startswith("__"):  # Ignore metadata keys
                value = data[key]
                if isinstance(value, np.ndarray) and value.dtype.names:
                        for name in value.dtype.names:
                            tmp_value =  data[key][0][name]
                            try:
                                tmp_value = tmp_value.item()
                                tmp_value = tmp_value.squeeze()
                                tmp_value = tmp_value.item()
                            except Exception as e:
                                pass
                            if ("{}_{}".format(key, name)) not in header_dict:
                                header_dict[("{}_{}".format(key, name))] = []
                            header_dict[("{}_{}".format(key, name))].append(tmp_value)
                else:
                    tmp_value = data[key]
                    try:
                        tmp_value = tmp_value.item()
                        tmp_value = tmp_value.squeeze()
                        tmp_value = tmp_value.item()
                    except Exception as e:
                        pass
                    if "{}".format(key) not in header_dict:
                        header_dict[("{}".format(key))] = []
                    header_dict[("{}".format(key))].append(tmp_value)

                    
    return header_dict

In [32]:
NLOS_header_dict = extract_header_info(NLOS_header_dict, NLOS_header_mat_files)
LOS_header_dict = extract_header_info(LOS_header_dict, LOS_header_mat_files)

In [33]:
NLOS_header_df = pd.DataFrame(NLOS_header_dict)
LOS_header_df = pd.DataFrame(LOS_header_dict)

In [34]:
def assert_errors(header_df):
    # Assert checks for data consistency
    assert header_df['TxData_lat_Deg'].nunique() == 1,"Error: Mupltiple TX latitudes detected"
    assert header_df['TxData_long_Deg'].nunique() == 1,"Error: Mupltiple TX longitudes detected" 
    assert header_df['TxData_alt'].nunique() == 1,"Error: Mupltiple TX altitudes detected" 

assert_errors(NLOS_header_df)
assert_errors(LOS_header_df)

In [35]:
def initialize_MPC(measurement_type):
    # Initialize MPC data structure
    mpc_dict = {"file_name":[], "mea_idx":[]}
    mpc_folder_path = "./data/measurements/Boulder_Downtown/Measurements/BoulderDowntown_28GHz_{}/MPC files/".format(measurement_type)  
    mpc_mat_files = glob.glob(f"{mpc_folder_path}/*.mat")  
    print(mpc_mat_files[:3]) 
    return mpc_dict, mpc_folder_path, mpc_mat_files

NLOS_mpc_dict, NLOS_mpc_folder_path, NLOS_mpc_mat_files = initialize_MPC(measurement_type1)
LOS_mpc_dict, LOS_mpc_folder_path, LOS_mpc_mat_files = initialize_MPC(measurement_type2)

['./data/measurements/Boulder_Downtown/Measurements/BoulderDowntown_28GHz_NLOS/MPC files/MPC2266.mat', './data/measurements/Boulder_Downtown/Measurements/BoulderDowntown_28GHz_NLOS/MPC files/MPC2267.mat', './data/measurements/Boulder_Downtown/Measurements/BoulderDowntown_28GHz_NLOS/MPC files/MPC2259.mat']
['./data/measurements/Boulder_Downtown/Measurements/BoulderDowntown_28GHz_LOS/MPC files/MPC1586.mat', './data/measurements/Boulder_Downtown/Measurements/BoulderDowntown_28GHz_LOS/MPC files/MPC1592.mat', './data/measurements/Boulder_Downtown/Measurements/BoulderDowntown_28GHz_LOS/MPC files/MPC1593.mat']


In [36]:
def format_MPC_dict(MPC_dict, MPC_mat_files):
    # MATLAB to Python data conversion
    for mat_file_path in MPC_mat_files: 

        #Load the .mat file
        data = loadmat(mat_file_path)

        # Store basic file info
        MPC_dict["file_name"].append(os.path.basename(mat_file_path))
        MPC_dict["mea_idx"].append(os.path.basename(mat_file_path)[3:3+4])

        # Recursive struct parsing
        for key in data:
            if not key.startswith("__"):  # Ignore metadata keys
                value = data[key]
                if isinstance(value, np.ndarray) and value.dtype.names:
                        for name in value.dtype.names:
                            tmp_value =  data[key][0][name]
                            try:
                                tmp_value = tmp_value.item()
                                tmp_value = tmp_value.squeeze()
                                tmp_value = tmp_value.item()
                            except Exception as e:
                                pass
                            if ("{}_{}".format(key, name)) not in MPC_dict:
                                MPC_dict[("{}_{}".format(key, name))] = []
                            MPC_dict[("{}_{}".format(key, name))].append(tmp_value)
                else:
                    tmp_value = data[key]
                    try:
                        tmp_value = tmp_value.item()
                        tmp_value = tmp_value.squeeze()
                        tmp_value = tmp_value.item()
                    except Exception as e:
                        pass
                    if "{}".format(key) not in MPC_dict:
                        MPC_dict[("{}".format(key))] = []
                    MPC_dict[("{}").format(key)].append(tmp_value)

                    
    return MPC_dict

NLOS_MPC_dict =  format_MPC_dict(NLOS_mpc_dict, NLOS_mpc_mat_files)
LOS_MPC_dict =  format_MPC_dict(LOS_mpc_dict, LOS_mpc_mat_files)


In [37]:
NLOS_MPC_df = pd.DataFrame(NLOS_MPC_dict)
print(NLOS_MPC_df.iloc[0].to_string())
print("\n")

LOS_MPC_df = pd.DataFrame(LOS_MPC_dict)

file_name                                                            MPC2266.mat
mea_idx                                                                     2266
header_TxGainEffective_dBi                                                     2
header_TxHPBW_Effective_deg                                                 47.3
header_dist_m                                                          61.906189
header_NumSectors                                                              8
header_B2B_Attenuation_dB                                                    -15
MPC_params_sector01            [[194.60000000000002, 133.19305332, 11.0, 129....
MPC_params_sector02            [[194.60000000000002, 127.69305331999999, 12.0...
MPC_params_sector03            [[194.625, 129.19305332, 11.5, 128.64943718814...
MPC_params_sector04            [[194.65, 127.69305331999999, 7.0, 128.4958396...
MPC_params_sector05            [[194.65, 131.19305332, 4.0, 128.0129647093495...
MPC_params_sector06         

In [38]:
NLOS_MPC_df

,file_name,mea_idx,header_TxGainEffective_dBi,header_TxHPBW_Effective_deg,header_dist_m,header_NumSectors,header_B2B_Attenuation_dB,MPC_params_sector01,MPC_params_sector02,MPC_params_sector03,MPC_params_sector04,MPC_params_sector05,MPC_params_sector06,MPC_params_sector07,MPC_params_sector08
0,MPC2266.mat,2266,2,47.3,61.906189,8,-15,"[[194.60000000000002, 133.19305332, 11.0, 129....","[[194.60000000000002, 127.69305331999999, 12.0...","[[194.625, 129.19305332, 11.5, 128.64943718814...","[[194.65, 127.69305331999999, 7.0, 128.4958396...","[[194.65, 131.19305332, 4.0, 128.0129647093495...","[[194.675, 128.69305332, 1.0, 127.612723886171...","[[194.70000000000002, 132.19305332, 1.0, 127.0...","[[194.75, 133.69305332, 1.5, 126.7786371010471..."
1,MPC2267.mat,2267,2,47.3,57.997207,8,-15,"[[191.35000000000002, 146.813161256, 8.0, 123....","[[191.4, 149.313161256, 8.0, 123.8883666412817...","[[191.35000000000002, 151.813161256, 8.0, 124....","[[191.3, 150.313161256, 8.0, 124.4827462855367...","[[191.25, 149.813161256, 8.0, 124.703082037615...","[[191.22500000000002, 147.313161256, 4.0, 124....","[[191.20000000000002, 147.313161256, 3.5, 124....","[[191.175, 147.313161256, 4.0, 124.03639160188..."
2,MPC2259.mat,2259,2,47.3,65.774953,8,-15,"[[223.0, 164.077449226, -13.0, 127.22384192359...","[[223.025, 172.077449226, 3.5, 131.63760982483...","[[224.45000000000002, 154.577449226, 8.0, 131....","[[222.85000000000002, 160.077449226, 8.0, 128....","[[222.875, 160.577449226, -7.0, 126.4211023219...","[[223.10000000000002, 160.577449226, -7.5, 127...","[[223.32500000000002, 165.577449226, 8.0, 129....","[[222.775, 162.077449226, -5.0, 125.8222808144..."
3,MPC2265.mat,2265,2,47.3,60.330643,8,-15,"[[197.47500000000002, 133.41385878, 2.5, 120.8...","[[197.425, 133.41385878, 3.5, 121.018505613346...","[[197.35000000000002, 135.41385878, 3.5, 121.0...","[[197.4, 133.41385878, 2.5, 120.73833756069689...","[[197.35000000000002, 133.41385878, 2.5, 120.9...","[[197.3, 135.41385878, -2.0, 120.9488232681919...","[[197.3, 133.41385878, -2.0, 120.8483776528297...","[[197.25, 133.41385878, -8.0, 120.430835527444..."
4,MPC2264.mat,2264,2,47.3,60.842554,8,-15,"[[199.925, 141.89762029899998, -12.5, 108.9974...","[[199.9, 142.89762029899998, -8.0, 109.7961630...","[[199.9, 142.89762029899998, -8.0, 109.9079264...","[[199.875, 142.89762029899998, -8.0, 109.96309...","[[199.925, 141.89762029899998, -13.0, 109.1272...","[[199.875, 142.89762029899998, -8.0, 110.03558...","[[199.875, 142.89762029899998, -8.0, 110.13829...","[[199.875, 142.89762029899998, -8.0, 110.16258..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
124,MPC2240.mat,2240,2,47.3,90.604338,8,-15,"[[362.15000000000003, 243.05277431099998, 8.0,...","[[362.15000000000003, 243.05277431099998, 12.5...","[[362.1, 243.05277431099998, 8.0, 131.31148706...","[[362.15000000000003, 248.05277431099998, 15.0...","[[364.77500000000003, 212.05277431099998, 8.0,...","[[364.8, 210.55277431099998, 8.0, 125.55023079...","[[303.5, 156.052774311, 2.0, 130.2356278263488...","[[303.5, 156.052774311, 2.0, 130.9542457492292..."
125,MPC2256.mat,2256,2,47.3,72.666507,8,-15,"[[237.775, 171.52549871399998, 2.0, 126.652712...","[[237.775, 172.02549871399998, 8.0, 126.891471...","[[237.8, 176.02549871399998, 8.0, 127.00736378...","[[237.82500000000002, 173.52549871399998, 12.5...","[[237.82500000000002, 173.52549871399998, 11.5...","[[237.85000000000002, 173.52549871399998, 10.0...","[[235.125, 158.02549871399998, -7.0, 132.00187...","[[235.125, 158.52549871399998, -13.5, 130.3811..."
126,MPC2242.mat,2242,2,47.3,88.023190,8,-15,"[[304.65000000000003, 144.90212413999996, 2.0,...","[[302.7, 155.40212413999996, -7.0, 131.6155110...","[[302.7, 167.40212413999996, -6.5, 132.4458895...","[[304.925, 136.40212413999996, -2.5, 129.07326...","[[304.825, 137.90212413999996, -2.0, 127.36889...","[[293.475, 167.40212413999996, 2.0, 131.962560...","[[293.375, 167.40212413999996, -1.0, 131.27550...","[[293.475, 163.9021241399

In [39]:
mpc_2266_df = NLOS_MPC_df.iloc[0]
mpc_2266_df = pd.DataFrame(mpc_2266_df).transpose()
mpc_2266_df

,file_name,mea_idx,header_TxGainEffective_dBi,header_TxHPBW_Effective_deg,header_dist_m,header_NumSectors,header_B2B_Attenuation_dB,MPC_params_sector01,MPC_params_sector02,MPC_params_sector03,MPC_params_sector04,MPC_params_sector05,MPC_params_sector06,MPC_params_sector07,MPC_params_sector08
0,MPC2266.mat,2266,2,47.3,61.906189,8,-15,"[[194.60000000000002, 133.19305332, 11.0, 129....","[[194.60000000000002, 127.69305331999999, 12.0...","[[194.625, 129.19305332, 11.5, 128.64943718814...","[[194.65, 127.69305331999999, 7.0, 128.4958396...","[[194.65, 131.19305332, 4.0, 128.0129647093495...","[[194.675, 128.69305332, 1.0, 127.612723886171...","[[194.70000000000002, 132.19305332, 1.0, 127.0...","[[194.75, 133.69305332, 1.5, 126.7786371010471..."


In [40]:
def cluster_rays(mpc_rx_df):
    flattened_rows = []

    # loop over sectors
    for sector in range(1, 9):
        col = f"MPC_params_sector{sector:02d}"
        
        if col not in mpc_rx_df.columns:
            continue  # skip missing sectors
        
        # loop over snapshots (rows in that column)
        for snapshot_idx, mpc_list in enumerate(mpc_rx_df[col]):
            if mpc_list is None:
                continue
            # loop over MPCs inside that snapshot
            for mpc in mpc_list:
                delay, az, el, pl = mpc
                flattened_rows.append({
                    "sector": sector,
                    "delay_ns": delay,
                    "azimuth_deg": az,
                    "elevation_deg": el,
                    "pathloss_dB": pl
                })

    # Build the final DataFrame
    flattened_df = pd.DataFrame(flattened_rows)

    # filter sector 1 only
    sector1_df = flattened_df[flattened_df["sector"] == 1]

    # sort values to compute differences
    sector1_sorted_delay = np.sort(sector1_df["delay_ns"].unique())
    sector1_sorted_az = np.sort(sector1_df["azimuth_deg"].unique())

    # compute consecutive differences
    delay_diffs = np.diff(sector1_sorted_delay)
    az_diffs = np.diff(sector1_sorted_az)

    # minimum non-zero tolerance
    min_delay_tol = delay_diffs[delay_diffs > 0].min()
    min_az_tol = az_diffs[az_diffs > 0].min()

    # clustering based on tolerance
    # Select features
    X = flattened_df[["azimuth_deg", "delay_ns"]].values

    # Define tolerances for each feature
    tolerance_az = min_az_tol    # degrees
    tolerance_delay = min_delay_tol # ns

    # Scale features so each unit = 1 tolerance
    X_scaled = np.zeros_like(X, dtype=float)
    X_scaled[:, 0] = X[:, 0] / tolerance_az
    X_scaled[:, 1] = X[:, 1] / tolerance_delay

    # Run hierarchical clustering
    # criterion='distance', t=1 ensures no cluster exceeds 1 scaled unit in any pairwise distance
    labels = fclusterdata(X_scaled, t=1, criterion='distance', metric='chebyshev', method='complete')

    # Add cluster labels to DataFrame
    flattened_df["cluster"] = labels

    # group by cluster and take the mean of each column
    cluster_means = flattened_df.groupby("cluster")[["delay_ns", "azimuth_deg", "elevation_deg", "pathloss_dB"]].mean()

    # convert to NumPy array
    cluster_means_array = cluster_means.to_numpy()
    return cluster_means_array, flattened_df

In [41]:
def add_MPC_cluster(mpc_df):
    # Create a new column for clustered MPCs
    mpc_df["clustered_MPCs"] = None
    
    for idx, row in mpc_df.iterrows():
        # Get cluster means and store them as a list
        cluster_means_array, flattened_df = cluster_rays(row.to_frame().transpose())
        mpc_df.at[idx, "clustered_MPCs"] = cluster_means_array.tolist()

    return mpc_df

NLOS_MPC_df = add_MPC_cluster(NLOS_MPC_df)
LOS_MPC_df = add_MPC_cluster(LOS_MPC_df)

In [42]:
NLOS_merged_df = pd.merge(NLOS_MPC_df, NLOS_header_df, on='mea_idx', how='inner')
LOS_merged_df = pd.merge(LOS_MPC_df, LOS_header_df, on='mea_idx', how='inner')

In [43]:
NLOS_merged_df.to_csv("NLOS_MPC_clustered.csv", index=False)
LOS_merged_df.to_csv("LOS_MPC_clustered.csv", index=False)

In [44]:
current_dir = os.getcwd()
print(current_dir)

/Users/davidlee/Desktop/Sionna_Princeton


In [45]:
# # get unique cluster labels
# clusters = flattened_df["cluster"].unique()

# plt.figure(figsize=(8,6))

# # plot each cluster separately with its own color
# for cluster in clusters:
#     cluster_points = flattened_df[flattened_df["cluster"] == cluster]
#     plt.scatter(cluster_points["delay_ns"], cluster_points["azimuth_deg"],
#                 s=10, label=f"Cluster {cluster}")

# plt.xlabel("Delay (ns)")
# plt.ylabel("Azimuth (°)")
# plt.title("MPC Clusters by Pathloss")
# plt.show()


In [46]:
# # count number of points in each cluster
# cluster_counts = flattened_df['cluster'].value_counts()

# # find the cluster label with the most points
# largest_cluster_label = cluster_counts.idxmax()

# # get all rows in the largest cluster
# largest_cluster_df = flattened_df[flattened_df['cluster'] == largest_cluster_label]

# print(f"Largest cluster: {largest_cluster_label}, size: {len(largest_cluster_df)}")
# largest_cluster_df.head()  # show first few rows


In [47]:
# flattened_df.groupby("cluster").agg(['min', 'max', 'count'])


In [49]:
# group by cluster and take the mean of each column
# cluster_means = flattened_df.groupby("cluster")[["azimuth_deg", "elevation_deg", "delay_ns", "pathloss_dB"]].mean()

# convert to NumPy array
# cluster_means_array = cluster_means.to_numpy()

# print(cluster_means_array.shape)   # (num_clusters, num_features)
# print(cluster_means_array)